In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

STUDENT_ID = "TUPM-26-1217"
SEED_DIGIT = int(STUDENT_ID[-1])
RANDOM_SEED = int("".join(STUDENT_ID.split("-")[1:]))

WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

measurement_path = DATA_DIR / "secom.data"

raw_measurements = np.genfromtxt(measurement_path, dtype=float)
missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)

all_missing = observed == 0
observed_min = np.where(missing, np.inf, raw_measurements).min(axis=0)
observed_max = np.where(missing, -np.inf, raw_measurements).max(axis=0)
constant = (observed > 0) & (observed_min == observed_max)

usable_indices = np.flatnonzero(~(all_missing | constant))

num_select = 5 + (SEED_DIGIT % 3)
rng = np.random.default_rng(seed=RANDOM_SEED)
selected_indices = np.sort(rng.choice(usable_indices, size=num_select, replace=False))

selected_data = raw_measurements[:, selected_indices].copy()

means = np.nanmean(selected_data, axis=0)
medians = np.nanmedian(selected_data, axis=0)
sample_sds = np.nanstd(selected_data, axis=0, ddof=1)
missing_pct = np.isnan(selected_data).mean(axis=0) * 100

mad = np.nanmedian(np.abs(selected_data - medians), axis=0)
safe_mad = np.where(mad == 0, np.nan, mad)
robust_z = (selected_data - medians) / safe_mad

flagged_mask = np.abs(robust_z) > 3.5
flagged_counts = flagged_mask.sum(axis=0)
flagged_pct = (flagged_counts / selected_data.shape[0]) * 100
total_flagged = int(flagged_mask.sum())

rank_order = np.argsort(flagged_pct)[::-1]

summary_df = pd.DataFrame({
    "Feature_Index": selected_indices[rank_order],
    "Mean": np.round(means[rank_order], 4),
    "Median": np.round(medians[rank_order], 4),
    "Sample_SD": np.round(sample_sds[rank_order], 4),
    "Missing_Pct": np.round(missing_pct[rank_order], 4),
    "MAD": np.round(mad[rank_order], 4),
    "Flagged_Count": flagged_counts[rank_order],
    "Flagged_Pct": np.round(flagged_pct[rank_order], 4)
})

csv_path = OUTPUT_DIR / "measurement_quality.csv"
summary_df.to_csv(csv_path, index=False)


top_feature = summary_df.iloc[0]
print("=== EXERCISE 1: VECTORIZED MEASUREMENT QUALITY ANALYSIS ===")
print("Selected Features               :", selected_indices.tolist())
print(f"Highest Missing Percentage      : {missing_pct.max():.4f}%")
print(f"Feature with Highest Flagged %  : Feature {int(top_feature['Feature_Index'])} ({top_feature['Flagged_Pct']:.4f}%)")
print("Total Flagged Observations      :", total_flagged)
print("Vectorized Results Verified     :", summary_df["Flagged_Count"].sum() == total_flagged)
print("\nRanked Summary Table:")
print(summary_df.to_string(index=False))

=== EXERCISE 1: VECTORIZED MEASUREMENT QUALITY ANALYSIS ===
Selected Features               : [55, 130, 318, 335, 490, 550]
Highest Missing Percentage      : 16.5922%
Feature with Highest Flagged %  : Feature 335 (14.7415%)
Total Flagged Observations      : 446
Vectorized Results Verified     : True

Ranked Summary Table:
 Feature_Index      Mean    Median  Sample_SD  Missing_Pct     MAD  Flagged_Count  Flagged_Pct
           335    3.4269    2.5135     5.7816       0.3829  0.5989            231      14.7415
           550   17.0133   16.3400     4.9670      16.5922  1.8100             84       5.3606
           490   51.3540   48.5574    18.0486       0.0638 11.0878             57       3.6375
           318    3.1888    2.8989     1.2159       0.0638  0.7504             54       3.4461
           130    0.7450    0.7588     0.0825       0.5743  0.0629             15       0.9572
            55 2856.1721 2854.0000    25.7493       0.2553 19.0000              5       0.3191
